In [1]:
import pandas as pd
import altair as alt

In [2]:
rust_data = pd.read_csv("./results.csv")
rust_data

,binary,file,filter,memory_peak_bytes,memory_total_bytes,size,time_seconds
0,Manual v1,photo-1547487452-5bdd9ce63723.jpg,ridge,20308077,2.030808e+07,small,192.000000
1,Manual v1,30ddbd81.jpg,ridge,16068612,1.606861e+07,small,153.000000
2,Manual v1,pfHrY2VQaGJuCsrFljKy_IMG_0257.JPG.jpg,ridge,4470340,4.470340e+06,small,40.000000
3,Manual v1,photo-1547413749-dd39525702cd.jpg,ridge,23980004,2.398000e+07,small,230.000000
4,Manual v1,photo-1544259342-306eccfec481.jpg,ridge,18852310,1.885231e+07,small,179.000000
...,...,...,...,...,...,...,...
2745,PyTorch,photo-1542291771-64f962701184.jpg,gaussian,85557248,2.851908e+07,medium,147.909264
2746,PyTorch,photo-1544647593-59b72ae685d5.jpg,gaussian,0,0.000000e+00,medium,120.150125
2747,PyTorch,photo-1543002601-b85cd5b8439b.jpg,gaussian,0,0.000000e+00,medium,234.304847
2748,PyTorch,photo-1546540551-29dc0a3f6786.jpg,gaussian,1310720,8.683520e+05,medium,96.361264


In [3]:
df_grouped = rust_data.drop(columns=["file"]).groupby(["binary", "filter", "size"]).mean().reset_index()
df_grouped

,binary,filter,size,memory_peak_bytes,memory_total_bytes,time_seconds
0,LLM v2,box-blur,large,19236022.60,1.923602e+07,158.760000
1,LLM v2,box-blur,medium,19260540.64,1.926054e+07,87.220000
2,LLM v2,box-blur,small,19286378.14,1.928638e+07,26.460000
3,LLM v2,gaussian,medium,19260540.64,1.926054e+07,84.500000
4,LLM v2,gaussian,small,19286378.14,1.928638e+07,25.820000
5,LLM v2,ridge,large,19208919.64,1.920892e+07,321.560000
6,LLM v2,ridge,medium,19260540.64,1.926054e+07,85.680000
7,LLM v2,ridge,small,19286378.14,1.928638e+07,27.200000
8,LLM v2,sharpen,large,19208919.64,1.920892e+07,322.380000
9,LLM v2,sharpen,medium,19260540.64,1.926054e+07,85.220000


In [4]:
# Step 2: Create a separate chart for each filter
for f in df_grouped["filter"].unique():
    chart = alt.Chart(df_grouped[df_grouped["filter"] == f]).mark_bar().encode(
        x=alt.X("size:N", sort=["small", "medium", "large"], title="Filter Size"),
        y=alt.Y("time_seconds:Q", title="Avg. Execution Time (ms)"),
        color=alt.Color("binary:N", title="Binary"),
        xOffset="binary:N"
    ).properties(
        width=300,
        height=300,
        title=f"Filter: {f}"
    ).configure_view(
        stroke=None  # removes outer border
    ).configure_axis(
        labelFontSize=12,
        titleFontSize=14,
        labelAngle=0
    ).configure_legend(
        titleFontSize=12,
        labelFontSize=12
    ).configure_title(
        fontSize=14
    ).configure_view(
        continuousHeight=300,
        continuousWidth=300
    )

    chart.save(f"./plots/time_all/{f}.png", ppi=300)
    chart.show()

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

In [5]:
# Step 2: Create a separate chart for each filter

no_pytorch = df_grouped[df_grouped["binary"] != "PyTorch"]

for f in no_pytorch["filter"].unique():
    chart = alt.Chart(no_pytorch[no_pytorch["filter"] == f]).mark_bar().encode(
        x=alt.X("size:N", sort=["small", "medium", "large"], title="Filter Size"),
        y=alt.Y("time_seconds:Q", title="Avg. Execution Time (ms)"),
        color=alt.Color("binary:N", title="Binary"),
        xOffset="binary:N"
    ).properties(
        width=300,
        height=300,
        title=f"Filter: {f}"
    ).configure_axis(
        labelFontSize=12,
        titleFontSize=14,
        labelAngle=0
    ).configure_legend(
        titleFontSize=12,
        labelFontSize=12
    ).configure_title(
        fontSize=14
    )

    chart.save(f"./plots/time_no_pt/{f}.png", ppi=300)
    chart.show()

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

In [6]:
# mem = df_grouped["memory_total_bytes"]
# df_grouped["mem_adv"] = (mem - mem.mean())

# Step 2: Create a separate chart for each filter
for f in df_grouped["filter"].unique():
    chart = alt.Chart(df_grouped[df_grouped["filter"] == f]).mark_bar().encode(
        x=alt.X("size:N", sort=["small", "medium", "large"], title="Filter Size"),
        y=alt.Y("memory_total_bytes:Q", title="Memory Advantage (bytes)"),
        color=alt.Color("binary:N", title="Binary"),
        xOffset="binary:N"
    ).properties(
        width=300,
        height=300,
        title=f"Filter: {f}"
    ).configure_axis(
        labelFontSize=12,
        titleFontSize=14,
        labelAngle=0
    ).configure_legend(
        titleFontSize=12,
        labelFontSize=12
    ).configure_title(
        fontSize=14
    )

    chart.save(f"./plots/memory/{f}.png", ppi=300)
    chart.show()

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

In [7]:
import pandas as pd
import altair as alt

# Define kernels
kernels = {
    "ridge": [
        [0.0, 0.0, -1.0, -1.0, 0.0, 0.0],
        [0.0, -1.0, -2.0, -2.0, -1.0, 0.0],
        [-1.0, -2.0, 7.0, 7.0, -2.0, -1.0],
        [-1.0, -2.0, 7.0, 7.0, -2.0, -1.0],
        [0.0, -1.0, -2.0, -2.0, -1.0, 0.0],
        [0.0, 0.0, -1.0, -1.0, 0.0, 0.0],
    ],
    "sharpen": [
        [0.0, 0.0, -1.0, -1.0, 0.0, 0.0],
        [0.0, -1.0, -2.0, -2.0, -1.0, 0.0],
        [-1.0, -2.0, 9.0, 9.0, -2.0, -1.0],
        [-1.0, -2.0, 9.0, 9.0, -2.0, -1.0],
        [0.0, -1.0, -2.0, -2.0, -1.0, 0.0],
        [0.0, 0.0, -1.0, -1.0, 0.0, 0.0],
    ],
    "box_blur": [[1.0 / 32.0] * 6 for _ in range(6)],
    "gaussian": [
        [0.00031, 0.00228, 0.00619, 0.00619, 0.00228, 0.00031],
        [0.00228, 0.01682, 0.04579, 0.04579, 0.01682, 0.00228],
        [0.00619, 0.04579, 0.12430, 0.12430, 0.04579, 0.00619],
        [0.00619, 0.04579, 0.12430, 0.12430, 0.04579, 0.00619],
        [0.00228, 0.01682, 0.04579, 0.04579, 0.01682, 0.00228],
        [0.00031, 0.00228, 0.00619, 0.00619, 0.00228, 0.00031],
    ]
}

# Generate and save charts
for name, matrix in kernels.items():
    df = pd.DataFrame([
        {"x": x, "y": y, "value": matrix[y][x]}
        for y in range(6)
        for x in range(6)
    ])
    
    chart = alt.Chart(df).mark_rect().encode(
        x=alt.X('x:O', axis=None),
        y=alt.Y('y:O', sort='descending', axis=None),
        color=alt.Color('value:Q', scale=alt.Scale(scheme='viridis'), legend=None)
    ).properties(
        width=150,
        height=150
    )
    
    chart.save(f"./plots/filters/{name}.png", ppi=300)


In [10]:
import torch

In [ ]:
q = lambda arr, l, :2 * torch.exp(-((len(arr) - l) ** 2) / (2 * 4 ** 2)) - 1 